In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# Define paths
processed_dir = r'A:\desktop\New folder\P\Ratul\Supply_Chain_Thesis\data\processed'
models_dir = r'A:\desktop\New folder\P\Ratul\Supply_Chain_Thesis\models'

print("Loading engineered datasets...")
X_train = pd.read_csv(os.path.join(processed_dir, 'X_train.csv'))
X_test = pd.read_csv(os.path.join(processed_dir, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(processed_dir, 'y_train.csv'))['days_for_shipping_real']
y_test = pd.read_csv(os.path.join(processed_dir, 'y_test.csv'))['days_for_shipping_real']

print(f"Data Loaded! X_train shape: {X_train.shape}")

Loading engineered datasets...
Data Loaded! X_train shape: (144415, 108)


In [2]:
# Identify the one-hot encoded shipping mode columns
shipping_cols = [col for col in X_train.columns if 'shipping_mode' in col]

# Calculate the median delivery time for each mode from the training data
baseline_rules = {}
for col in shipping_cols:
    median_val = y_train[X_train[col] == 1].median()
    baseline_rules[col] = median_val

# Function to apply these rules to the test set
def get_baseline_pred(row):
    for col in shipping_cols:
        if row[col] == 1:
            return baseline_rules[col]
    return y_train.median() # Fallback if mode is missing

baseline_preds = X_test.apply(get_baseline_pred, axis=1)
baseline_mae = mean_absolute_error(y_test, baseline_preds)

print(f"Valid Heuristic Baseline MAE: {baseline_mae:.3f} days")

Valid Heuristic Baseline MAE: 0.978 days


In [3]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

# 1. Define the Base Models (XGBoost and LightGBM)
base_models = [
    ('xgb', xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ('lgbm', lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1))
]

# 2. Define the Meta-Learner (Ridge Regression)
meta_learner = Ridge(alpha=1.0)

# 3. Build the Stacking Ensemble
stacked_ensemble = StackingRegressor(estimators=base_models, final_estimator=meta_learner, cv=5)

print("Training the Stacked Ensemble (XGBoost + LightGBM -> Ridge Meta-Learner)...")
stacked_ensemble.fit(X_train, y_train)

# Predict and Evaluate
stacked_preds = stacked_ensemble.predict(X_test)
stacked_mae = mean_absolute_error(y_test, stacked_preds)

print(f"Final Stacked Ensemble Test Set MAE: {stacked_mae:.3f} days")

# Export for Production
joblib.dump(stacked_ensemble, os.path.join(models_dir, 'production_stacked_model.pkl'))

Training the Stacked Ensemble (XGBoost + LightGBM -> Ridge Meta-Learner)...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003537 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 981
[LightGBM] [Info] Number of data points in the train set: 144415, number of used features: 107
[LightGBM] [Info] Start training from score 3.498085
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003543 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 959
[LightGBM] [Info] Number of data points in the train set: 115532, number of used features: 107
[Li

['A:\\desktop\\New folder\\P\\Ratul\\Supply_Chain_Thesis\\models\\production_stacked_model.pkl']

In [4]:
alphas = [0.1, 0.5, 0.9] # 10th percentile, Median, 90th percentile
quantile_preds = {}

print("Training LightGBM Quantile Models for Probabilistic Forecasting...")
for alpha in alphas:
    # Objective set to 'quantile' enables uncertainty bands
    model_q = lgb.LGBMRegressor(objective='quantile', alpha=alpha, metric='quantile', n_estimators=100, random_state=42, n_jobs=-1)
    model_q.fit(X_train, y_train)
    quantile_preds[alpha] = model_q.predict(X_test)

# Combine into a single DataFrame to view the best/expected/worst case scenarios
probabilistic_forecast = pd.DataFrame({
    'Actual_Days': y_test.values,
    'Optimistic_ETA_(10%)': quantile_preds[0.1],
    'Expected_ETA_(50%)': quantile_preds[0.5],
    'Pessimistic_ETA_(90%)': quantile_preds[0.9]
})

print("\nProbabilistic Forecast Sample (Uncertainty Bands):")
print(probabilistic_forecast.head(10))

Training LightGBM Quantile Models for Probabilistic Forecasting...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004737 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 981
[LightGBM] [Info] Number of data points in the train set: 144415, number of used features: 107
[LightGBM] [Info] Start training from score 2.000000
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003706 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 981
[LightGBM] [Info] Number of data points in the train set: 144415, number of used features: 107
[LightGBM] [

In [5]:
# Ensure shipping_cols is defined for the error analysis
shipping_cols = [col for col in X_train.columns if 'shipping_mode' in col]

# Create a dataframe of actuals vs predictions using the STACKED ENSEMBLE predictions
error_df = pd.DataFrame({
    'Actual_Delay': y_test.values,
    'Predicted_Delay': stacked_preds,
    'Absolute_Error': abs(y_test.values - stacked_preds)
})

# Reverse-engineer the one-hot encoded shipping mode to see where the model struggles
error_df['Shipping_Mode'] = X_test[shipping_cols].idxmax(axis=1).apply(lambda x: x.replace('shipping_mode_', '')).values

print("Mean Absolute Error by Shipping Mode (Stacked Ensemble):")
print(error_df.groupby('Shipping_Mode')['Absolute_Error'].mean().sort_values(ascending=False))

print("\nError analysis complete. The Stacked Ensemble framework is fully evaluated and exported.")

Mean Absolute Error by Shipping Mode (Stacked Ensemble):
Shipping_Mode
Standard Class    1.196481
Second Class      1.179037
Same Day          0.493705
First Class       0.022812
Name: Absolute_Error, dtype: float64

Error analysis complete. The Stacked Ensemble framework is fully evaluated and exported.


In [6]:
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import numpy as np
import os

print("--- Running RP Baseline Models ---")

# 1. Rule-Based Distance Heuristic (RB-DH) Baseline
rb_dh_mapping = y_train.groupby(X_train['shipping_mode_Standard Class']).mean().to_dict() 
rb_preds = X_test['shipping_mode_Standard Class'].map(rb_dh_mapping).fillna(y_train.mean())
rb_mae = mean_absolute_error(y_test, rb_preds)
print(f"Rule-Based Heuristic (RB-DH) MAE: {rb_mae:.3f} days")

# 2. ARIMA Time Series Baseline
# Since 'df' is not in memory, we use the chronologically sorted y_train.
# We aggregate it into sequential chunks (e.g., 500 orders per time step) to create a valid time series
chunk_size = 500
time_series_train = y_train.groupby(np.arange(len(y_train)) // chunk_size).mean()

print("Training ARIMA model (this might take a few seconds)...")
arima_model = sm.tsa.ARIMA(time_series_train.values, order=(2,1,2)).fit()

# Predict the baseline average for the test set duration
arima_preds = [arima_model.forecast(1)[0]] * len(y_test) 
arima_mae = mean_absolute_error(y_test, arima_preds)
print(f"ARIMA (2,1,2) Baseline MAE: {arima_mae:.3f} days")

print(f"\n--- Running RP Advanced Evaluation ---")

# 3. Decision Curve Analysis (DCA) - Net Benefit for > 3 Days threshold
threshold = 3.0
y_test_bin = (y_test > threshold).astype(int)
preds_bin = (stacked_preds > threshold).astype(int)

true_positives = ((preds_bin == 1) & (y_test_bin == 1)).sum()
false_positives = ((preds_bin == 1) & (y_test_bin == 0)).sum()
n = len(y_test_bin)
threshold_prob = 0.5 

net_benefit = (true_positives / n) - (false_positives / n) * (threshold_prob / (1 - threshold_prob))
print(f"Decision Curve Net Benefit (Threshold > 3 days): {net_benefit:.4f}")

# 4. Expected Calibration Error (ECE) Proxy for Regression
residuals = abs(y_test.values - stacked_preds)
ece_proxy = residuals.mean() / y_test.mean() * 100
print(f"Regression Calibration Error (Relative): {ece_proxy:.2f}%")

print(f"\n--- Generating Final Reports in /reports folder ---")
os.makedirs('../reports/figures', exist_ok=True)

# Save Final Performance Report
report_path = '../reports/final_model_performance.txt'
with open(report_path, 'w') as f:
    f.write("=== FINAL SUPPLY CHAIN MODEL PERFORMANCE ===\n")
    f.write(f"Stacked Ensemble MAE: {stacked_mae:.3f} days\n")
    f.write(f"Baseline RB-DH MAE: {rb_mae:.3f} days\n")
    f.write(f"Baseline ARIMA MAE: {arima_mae:.3f} days\n")
    f.write(f"Decision Curve Net Benefit (>3 days): {net_benefit:.4f}\n")
    f.write(f"Calibration Error: {ece_proxy:.2f}%\n")

# Save a comparison chart to reports/figures/
plt.figure(figsize=(8, 5))
models = ['ARIMA', 'RB-DH Heuristic', 'Stacked Ensemble']
maes = [arima_mae, rb_mae, stacked_mae]
plt.bar(models, maes, color=['red', 'orange', 'green'])
plt.title('Model Performance Comparison (Lower is Better)')
plt.ylabel('Mean Absolute Error (Days)')
for i, v in enumerate(maes):
    plt.text(i, v + 0.05, f"{v:.2f}", ha='center')
plt.savefig('../reports/figures/model_comparison_bar.png', bbox_inches='tight')
plt.close()

print(f"Reports successfully written to {report_path} and figures saved!")

--- Running RP Baseline Models ---
Rule-Based Heuristic (RB-DH) MAE: 1.252 days
Training ARIMA model (this might take a few seconds)...
ARIMA (2,1,2) Baseline MAE: 1.420 days

--- Running RP Advanced Evaluation ---
Decision Curve Net Benefit (Threshold > 3 days): 0.1558
Regression Calibration Error (Relative): 27.88%

--- Generating Final Reports in /reports folder ---
Reports successfully written to ../reports/final_model_performance.txt and figures saved!


In [7]:

print("Generating final RP visual charts (DCA and Calibration)...")

# 1. Regression Calibration Chart (Dependability)
plt.figure(figsize=(8, 6))
# Create 10 bins of predictions to see if predicted delay matches actual delay
bins = np.linspace(min(stacked_preds), max(stacked_preds), 10)
binned_true = np.digitize(stacked_preds, bins)
bin_means_pred = [stacked_preds[binned_true == i].mean() for i in range(1, len(bins))]
bin_means_true = [y_test.values[binned_true == i].mean() for i in range(1, len(bins))]

plt.plot(bin_means_pred, bin_means_true, marker='o', color='blue', label='Model Calibration')
plt.plot([min(stacked_preds), max(stacked_preds)], [min(stacked_preds), max(stacked_preds)], 
         linestyle='--', color='gray', label='Perfect Calibration')
plt.title('Calibration Chart of Dependability (Predicted vs Actual)')
plt.xlabel('Predicted Lead Time (Days)')
plt.ylabel('Actual Lead Time (Days)')
plt.legend()
plt.savefig('../reports/figures/calibration_chart.png', bbox_inches='tight')
plt.close()

# 2. Decision Curve Analysis (DCA) Chart
plt.figure(figsize=(8, 6))
threshold_probs = np.linspace(0.1, 0.9, 9)
net_benefits = []

for pt in threshold_probs:
    tp = ((preds_bin == 1) & (y_test_bin == 1)).sum()
    fp = ((preds_bin == 1) & (y_test_bin == 0)).sum()
    nb = (tp / n) - (fp / n) * (pt / (1 - pt))
    net_benefits.append(max(0, nb)) # Floor at 0 for standard DCA plotting

plt.plot(threshold_probs, net_benefits, marker='s', color='purple', label='Stacked Ensemble Net Benefit')
plt.axhline(0, color='black', linestyle='--', label='Treat None (Baseline)')
plt.title('Decision Curve Analysis (DCA) for >3 Day Delay Risk')
plt.xlabel('Threshold Probability')
plt.ylabel('Net Benefit')
plt.legend()
plt.savefig('../reports/figures/dca_curve.png', bbox_inches='tight')
plt.close()

print("Charts successfully saved to reports/figures/dca_curve.png and calibration_chart.png!")

Generating final RP visual charts (DCA and Calibration)...


C:\Users\manwa_vjycapl\AppData\Local\Temp\ipykernel_28124\339014031.py:8: RuntimeWarning: Mean of empty slice
  bin_means_pred = [stacked_preds[binned_true == i].mean() for i in range(1, len(bins))]
c:\Users\manwa_vjycapl\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\manwa_vjycapl\AppData\Local\Temp\ipykernel_28124\339014031.py:9: RuntimeWarning: Mean of empty slice
  bin_means_true = [y_test.values[binned_true == i].mean() for i in range(1, len(bins))]


Charts successfully saved to reports/figures/dca_curve.png and calibration_chart.png!
